# أنماط العودية المتقدمة في UnifyWeaver

يوضح هذا الدفتر من دفاتر الملاحظات أنماط العودية الأربعة الرئيسية التي يمكن لـ UnifyWeaver اكتشافها وتحسينها:

1. **عودية الذيل (Tail Recursion)** - حلقات تكرارية باستخدام مجمعات القيمة (accumulators)
2. **العودية الخطية (Linear Recursion)** - استدعاء عودي واحد مع الحفظ في الذاكرة (memoization)
3. **العودية الشجرية (Tree Recursion)** - استدعاءات عودية متعددة على أجزاء الهيكل
4. **العودية المتبادلة (Mutual Recursion)** - محددات تستدعي بعضها البعض في دورات

## الأهداف التعليمية

- فهم أنماط العودية المختلفة
- معرفة كيف يكتشف UnifyWeaver كل نمط ويحسنه
- مقارنة خصائص الأداء
- معرفة متى تستخدم كل نمط

## الإعداد

تهيئة بيئة UnifyWeaver.

In [ ]:
% Load initialization
['../init'].

% Load necessary modules
use_module(unifyweaver(core/recursive_compiler)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## النمط 1: عودية الذيل (Tail Recursion)

تستخدم عودية الذيل مجمعًا (accumulator) لنقل النتائج الوسيطة، ويكون الاستدعاء العودي هو **الإجراء الأخير** في الدالة.

### مثال: عد العناصر في قائمة

In [ ]:
% Define tail-recursive count_items
:- dynamic count_items/3.

% Base case: empty list, return accumulator
count_items([], Acc, Acc).

% Recursive case: increment accumulator, recurse on tail
count_items([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count_items(T, Acc1, N).  % ← Tail position!

### الاختبار في Prolog

In [ ]:
% Test: count items in [a,b,c,d,e]
\+ \+ (
    count_items([a,b,c,d,e], 0, _N),
    format('Count: ~w~n', [_N])
).

### التحقق من اكتشاف النمط

In [ ]:
% Check if detected as tail recursive
\+ \+ (
    is_tail_recursive_accumulator(count_items/3, _AccInfo),
    format('Tail recursive: ~w~n', [_AccInfo])
).

### التجميع إلى Bash

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(count_items/3, [], _BashCode),
    setup_call_cleanup(
        open('../output/count_items_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled count_items to Bash with tail recursion optimization')
).

### اختبار كود Bash المولد

In [ ]:
%%bash
source ../output/count_items_demo.sh
echo "Counting items in [a,b,c,d,e]:"
count_items "[a,b,c,d,e]" 0 ""

## النمط 2: العودية الخطية (Linear Recursion)

تحتوي العودية الخطية على استدعاء عودي **واحد فقط** لكل بند، مع حدوث العمليات الحسابية بعد عودة الاستدعاء العودي.

### مثال: العاملي (Factorial)

In [ ]:
% Define factorial
:- dynamic factorial/2.

% Base case
factorial(0, 1).

% Recursive case: exactly ONE recursive call
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),  % ← One recursive call
    F is N * F1.        % ← Computation after call

### الاختبار في Prolog

In [ ]:
% Test: factorial of 5
\+ \+ (
    factorial(5, _F),
    format('5! = ~w~n', [_F])
).

### التحقق من اكتشاف النمط

In [ ]:
% Check if detected as linear recursive
is_linear_recursive_streamable(factorial/2),
writeln('✓ Detected as linear recursion').

### التجميع إلى Bash

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(factorial/2, [], _BashCode),
    % Keep function definitions only; Brush treats sourced scripts as direct execution
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Auto-execute when run directly (not when sourced)"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/factorial_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled factorial to Bash with fold-based linear recursion')
).

### اختبار كود Bash المولد

In [ ]:
%%bash
source ../output/factorial_demo.sh
echo "Factorial of 5:"
factorial 5 ""
echo ""
echo "Factorial of 10:"
factorial 10 ""

## النمط 3: العودية الشجرية (Tree Recursion)

تُجري العودية الشجرية استدعاءات عودية **متعددة** لمعالجة أجزاء مختلفة من الهيكل البرمجي.

### مثال: مجموع الشجرة (Tree Sum)

In [ ]:
% Define tree_sum for binary trees
% Tree format: [Value, LeftSubtree, RightSubtree] or []
:- dynamic tree_sum/2.

% Base case: empty tree has sum 0
tree_sum([], 0).

% Recursive case: sum = value + left_sum + right_sum
tree_sum([V, L, R], Sum) :-
    tree_sum(L, LS),   % ← First recursive call
    tree_sum(R, RS),   % ← Second recursive call
    Sum is V + LS + RS.

### الاختبار في Prolog

In [ ]:
% Test: tree_sum of [5, [3, [1, [], []], []], [2, [], []]]
%       5
%      / \
%     3   2
%    /
%   1
\+ \+ (
    tree_sum([5, [3, [1, [], []], []], [2, [], []]], _Sum),
    format('Tree sum: ~w (expected 11)~n', [_Sum])
).

### التجميع إلى Bash

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(tree_sum/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/tree_sum_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled tree_sum to Bash with tree recursion')
).

### اختبار كود Bash المولد

In [ ]:
%%bash
source ../output/tree_sum_demo.sh
echo "Tree sum of [5,[3,[1,[],[]],[]],[2,[],[]]]:"
tree_sum "[5,[3,[1,[],[]],[]],[2,[],[]]]"

## النمط 4: العودية المتبادلة (Mutual Recursion)

تحدث العودية المتبادلة عندما يستدعي محددان أو أكثر بعضهما البعض في دورة متكررة.

### مثال: الزوجي (Even) والفردي (Odd)

In [ ]:
% Define mutually recursive is_even and is_odd
:- dynamic is_even/1.
:- dynamic is_odd/1.

% is_even base case
is_even(0).

% is_even recursive: N is even if N-1 is odd
is_even(N) :-
    N > 0,
    N1 is N - 1,
    is_odd(N1).  % ← Calls is_odd

% is_odd base case
is_odd(1).

% is_odd recursive: N is odd if N-1 is even
is_odd(N) :-
    N > 1,
    N1 is N - 1,
    is_even(N1).  % ← Calls is_even

### الاختبار في Prolog

In [ ]:
% Test even/odd
is_even(0), writeln('✓ 0 is even').
is_even(4), writeln('✓ 4 is even').
is_odd(3), writeln('✓ 3 is odd').
is_odd(7), writeln('✓ 7 is odd').

### التحقق من العودية المتبادلة

In [ ]:
% Build call graph and find SCCs
\+ \+ (
    use_module(unifyweaver(core/advanced/call_graph)),
    use_module(unifyweaver(core/advanced/scc_detection)),

    build_call_graph([is_even/1, is_odd/1], _Graph),
    format('Call graph: ~w~n', [_Graph]),

    find_sccs(_Graph, _SCCs),
    format('SCCs (mutual recursion groups): ~w~n', [_SCCs])
).

### التجميع إلى Bash

In [ ]:
% Compile the mutual recursion group
\+ \+ (
    use_module(unifyweaver(core/advanced/mutual_recursion)),

    compile_mutual_recursion([is_even/1, is_odd/1], [], _BashCode),
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Main dispatch: route command line calls to functions"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/even_odd_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled is_even/is_odd to Bash with shared memoization')
).

### اختبار كود Bash المولد

In [ ]:
%%bash
source ../output/even_odd_demo.sh
echo "Testing is_even and is_odd:"
is_even 0 >/dev/null && echo "✓ 0 is even"
is_even 4 >/dev/null && echo "✓ 4 is even"
is_odd 3 >/dev/null && echo "✓ 3 is odd"
is_odd 7 >/dev/null && echo "✓ 7 is odd"
is_even 5 >/dev/null 2>&1 || echo "✓ 5 is not even"

## مقارنة الأنماط

دعنا نقارن خصائص كل نمط:

| النمط | الاستدعاءات العودية | التحسين | التعقيد المكاني | الأفضل لـ |
|:--------|:----------------|:-------------|:-----------------|:---------|
| **عودية الذيل** | 1 (في موضع الذيل) | حلقة تكرارية | O(1) | المجمعات، المسح الخطي |
| **العودية الخطية** | 1 (في أي موضع) | طي (Fold) + حفظ في الذاكرة | O(n) لجدول الحفظ | فيبوناتشي، العاملي |
| **العودية الشجرية** | 2+ (على أجزاء الهيكل) | التفكيك الهيكلي | O(depth) لحجم المكدس | عمليات الأشجار والرسوم البيانية |
| **العودية المتبادلة** | 1+ (عبر المحددات) | حفظ مشترك في الذاكرة | O(n) للجدول المشترك | زوجي/فردي، التعريفات المتبادلة |

## ترتيب اكتشاف الأنماط

يحاول UnifyWeaver مطابقة الأنماط بهذا الترتيب:

1. **عودية الذيل** (الأكثر كفاءة)
2. **العودية الخطية** (ما لم تكن محظورة)
3. **العودية الشجرية** (الهيكلية)
4. **العودية المتبادلة** (اكتشاف المكونات شديدة الترابط SCC)
5. **العودية الأساسية** (البديل التلقائي)

يمكنك التأثير على عملية الاكتشاف باستخدام `forbid_linear_recursion/1`.

## تمرين: دورك الآن!

جرب تعريف هذه المحددات وتجميعها:

### 1. مجموع القائمة بعودية الذيل
```prolog
sum_list([], Acc, Acc).
sum_list([H|T], Acc, Sum) :-
    Acc1 is Acc + H,
    sum_list(T, Acc1, Sum).
```

### 2. فيبوناتشي بالعودية الخطية
```prolog
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.
```

### 3. ارتفاع الشجرة
```prolog
tree_height([], 0).
tree_height([_, L, R], H) :-
    tree_height(L, HL),
    tree_height(R, HR),
    H is max(HL, HR) + 1.
```

In [ ]:
% Your code here!


## الملخص

في هذا الدفتر، تعلمت:

✅ أنماط العودية الأربعة الرئيسية في UnifyWeaver

✅ كيفية تعريف كل نمط في Prolog

✅ كيف يكتشف UnifyWeaver كل نمط ويحسنه

✅ خصائص الأداء لكل نمط

✅ متى تستخدم كل نمط

## الخطوات التالية

تابع إلى **دفتر الملاحظات 3: العرض المرئي لرسم الاستدعاءات** للتعرف على التحليل البرمجي المتقدم والعرض المرئي!